# Detección de artefactos en ECG con Random Cut Forest

**Entorno**: SageMaker **Notebook Instance**  
**Algoritmo**: Random Cut Forest (RCF) — built-in de SageMaker, **no supervisado** (detección de anomalías).  
**Dataset**: archivos CSV en la carpeta local `./dataset/` (salida de `generate_ecg_dataset.py`).  
**Rol de ejecución**: el de la Notebook Instance (auto-detectado) o el que definas en la variable `EXECUTION_ROLE_ARN`.

## Planteamiento del problema

Queremos detectar **artefactos** en señales ECG (pérdida de contacto del electrodo, ruido muscular, artefactos de movimiento, deriva de línea base). En el mundo real no existe una etiqueta fiable de "artefacto" para la mayoría de los datos históricos, por eso usamos un enfoque **no supervisado**: el algoritmo aprende cómo luce un latido "normal" y asigna un **anomaly score** más alto a los latidos que se desvían de ese patrón.

En este dataset sintético **sí** tenemos etiqueta (`is_artifact`), pero **no se usa durante el entrenamiento**. Solo la usamos al final para evaluar el modelo con ROC, precision/recall, matriz de confusión y optimización de umbral.

## ¿Por qué Random Cut Forest?

RCF construye un conjunto de árboles partiendo el espacio de features por cortes aleatorios. Un punto que requiere **muy pocos cortes** para quedar aislado es probablemente una anomalía. Su anomaly score es el promedio de profundidad de aislamiento a lo largo del ensemble. Ventajas:

- Maneja datos de alta dimensión.
- No asume distribución gaussiana.
- Robusto a outliers legítimos vs artefactos.
- Disponible como algoritmo built-in en SageMaker (no hay que escribir código de training).

## Prerrequisitos

1. **Notebook Instance** `ml.t3.medium` con un rol IAM que tenga permisos de SageMaker + S3 + CloudWatch Logs.
2. Subir al directorio raíz del notebook instance:
   - Este `.ipynb`
   - Carpeta `dataset/` con `train.csv` y `validation.csv` adentro.
3. Kernel: `conda_python3`.

### Sobre credenciales
La Notebook Instance hereda las credenciales del rol adjunto automáticamente. **No pegues claves en ninguna celda**.

## 0. Instalar dependencias

Solo instala lo que falte (típicamente `seaborn`). No fuerza upgrades para evitar recompilar numpy con el GCC viejo de la Notebook Instance.

In [ ]:
import sys
import subprocess
import importlib

def ensure(pkg_pip, import_name=None):
    mod = import_name or pkg_pip
    try:
        importlib.import_module(mod)
        print(f'  ✓ {mod} ya instalado')
    except ImportError:
        print(f'  ↻ Instalando {pkg_pip}...')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '--quiet', pkg_pip
        ])

required = [
    ('boto3', 'boto3'),
    ('sagemaker', 'sagemaker'),
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('scikit-learn', 'sklearn'),
]
for pkg, mod in required:
    ensure(pkg, mod)

print('\n✓ Dependencias listas.')

## 1. Setup del entorno SageMaker

In [ ]:
import boto3
import sagemaker
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sagemaker import RandomCutForest

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# ---------------------------------------------------------------------------
# Configuración del entorno — agnóstico de cuenta.
# Si ejecutas esto en una Notebook Instance, deja EXECUTION_ROLE_ARN en None
# y se detecta automáticamente. Para forzar un rol específico, ponlo aquí.
# ---------------------------------------------------------------------------
EXECUTION_ROLE_ARN = None   # ej: 'arn:aws:iam::123456789012:role/MiRol'

session = sagemaker.Session()
region = session.boto_region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

if EXECUTION_ROLE_ARN:
    role = EXECUTION_ROLE_ARN
else:
    role = sagemaker.get_execution_role()

try:
    bucket = session.default_bucket()
except Exception as e:
    bucket = f'sagemaker-{region}-{account_id}'
    print(f'default_bucket() falló ({e}); usando {bucket}')
    s3 = boto3.client('s3', region_name=region)
    try:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket)
        else:
            s3.create_bucket(
                Bucket=bucket,
                CreateBucketConfiguration={'LocationConstraint': region}
            )
    except s3.exceptions.BucketAlreadyOwnedByYou:
        pass
    except Exception as e2:
        print(f'Nota: no se pudo crear el bucket ({e2}).')

prefix = 'ecg-artifacts-rcf'

print(f'Cuenta AWS:  {account_id}')
print(f'Región:      {region}')
print(f'Rol:         {role}')
print(f'Bucket:      s3://{bucket}/{prefix}/')

## 2. Cargar y explorar el dataset

Los archivos están en `./dataset/`. Hacemos una exploración rápida para entender distribuciones y confirmar que los artefactos son distinguibles antes de entrenar.

In [ ]:
from pathlib import Path

DATASET_DIR = Path('dataset')
train_file = DATASET_DIR / 'train.csv'
val_file = DATASET_DIR / 'validation.csv'

assert train_file.exists(), f'No se encuentra {train_file}. Sube la carpeta dataset/.'
assert val_file.exists(), f'No se encuentra {val_file}.'

train_df = pd.read_csv(train_file)
val_df = pd.read_csv(val_file)

print(f'Train: {train_df.shape}   Artefactos: {train_df["is_artifact"].sum()} '
      f'({train_df["is_artifact"].mean()*100:.2f}%)')
print(f'Val:   {val_df.shape}   Artefactos: {val_df["is_artifact"].sum()} '
      f'({val_df["is_artifact"].mean()*100:.2f}%)')
train_df.head()

In [ ]:
# Visualización de distribuciones por clase
numeric_features = [
    'rr_interval_ms', 'pr_interval_ms',
    'p_wave_amplitude_mv', 't_wave_amplitude_mv',
    'morphology_score', 'heart_rate_bpm'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.ravel(), numeric_features):
    sns.kdeplot(data=train_df, x=feat, hue='is_artifact',
                ax=ax, common_norm=False, palette={0: '#27ae60', 1: '#c0392b'},
                fill=True, alpha=0.35)
    ax.set_title(feat)
fig.suptitle('Distribuciones por clase (0 = limpio, 1 = artefacto)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 3. Preparar datos para RCF

**Decisiones importantes:**

- **Dropeamos `timestamp`**: es un string ISO; no es una feature numérica.
- **Dropeamos `time_offset_seconds`**: monótono creciente — no aporta señal de anomalía.
- **Dropeamos `is_artifact`**: RCF es **no supervisado**; la etiqueta solo se usa después para evaluar.
- **Dropeamos `heart_rate_bpm`**: redundante con `rr_interval_ms` (correlación perfecta). Dejar ambos infla la dimensionalidad sin información nueva.

Features finales para RCF: `rr_interval_ms`, `pr_interval_ms`, `p_wave_amplitude_mv`, `t_wave_amplitude_mv`, `morphology_score`.

In [ ]:
RCF_FEATURES = [
    'rr_interval_ms',
    'pr_interval_ms',
    'p_wave_amplitude_mv',
    't_wave_amplitude_mv',
    'morphology_score',
]

# Arrays float32 como espera RCF
X_train = train_df[RCF_FEATURES].values.astype('float32')
X_val = val_df[RCF_FEATURES].values.astype('float32')
y_val = val_df['is_artifact'].values.astype('int32')

print(f'X_train shape: {X_train.shape}')
print(f'X_val   shape: {X_val.shape}')
print(f'y_val   shape: {y_val.shape}  (solo para evaluación)')
print(f'\nFeatures: {RCF_FEATURES}')

## 4. Configurar y entrenar Random Cut Forest

### Hiperparámetros clave
| Parámetro | Valor | Por qué |
|---|---|---|
| `num_samples_per_tree` | 256 | Tamaño muestra por árbol. 256 es estándar; si aumentas, más memoria y cómputo. |
| `num_trees` | 100 | Número de árboles en el ensemble. Más árboles → scores más estables pero más lento. |
| `feature_dim` | 5 | Se setea automáticamente al llamar `record_set`. |

RCF se entrena con el método `record_set()` que convierte los arrays numpy al formato protobuf esperado por el contenedor. Si pasamos dos record sets (train + test con labels), SageMaker reporta F1 en el test automáticamente.

In [ ]:
rcf = RandomCutForest(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',   # si falla cuota, prueba ml.m4.xlarge
    data_location=f's3://{bucket}/{prefix}/',
    output_path=f's3://{bucket}/{prefix}/output',
    num_samples_per_tree=256,
    num_trees=100,
    base_job_name='ecg-rcf',
    sagemaker_session=session,
)

In [ ]:
# Construir los record_set y lanzar el training.
# El canal 'test' con labels permite que SageMaker reporte F1 durante el entrenamiento.
train_records = rcf.record_set(X_train, channel='train')
test_records = rcf.record_set(X_val, labels=y_val.astype('float32'), channel='test')

rcf.fit([train_records, test_records], wait=True)

## 5. Desplegar el endpoint para inferencia

Al consultar el endpoint con un vector de features, RCF devuelve el **anomaly score**. No es una probabilidad entre 0 y 1: es una medida sin escala fija, donde valores más altos indican mayor anomalía. El umbral se elige analizando la distribución (o, si tenemos etiquetas como aquí, con ROC/F1).

In [ ]:
predictor = rcf.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
)
print(f'Endpoint: {predictor.endpoint_name}')

In [ ]:
# Inferencia sobre el set de validación
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

def get_scores(X, batch_size=100):
    all_scores = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        response = predictor.predict(batch)
        batch_scores = [float(p['score']) for p in response['scores']]
        all_scores.extend(batch_scores)
    return np.array(all_scores)

scores = get_scores(X_val)
print(f'Scores obtenidos: {len(scores)}')
print(f'Rango: [{scores.min():.4f}, {scores.max():.4f}]   Media: {scores.mean():.4f}')

## 6. Métricas de calidad

Aquí está la parte interesante. Como tenemos ground truth (`y_val`), podemos evaluar el modelo igual que un clasificador binario, usando el anomaly score como predictor continuo.

### 6.1 Distribución de anomaly scores

Si el modelo funciona bien, la distribución de scores de los artefactos (rojo) debería estar desplazada a la derecha respecto a la de los latidos limpios (verde).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores[y_val == 0], bins=40, alpha=0.6, label='Limpio (0)', color='#27ae60')
ax.hist(scores[y_val == 1], bins=40, alpha=0.6, label='Artefacto (1)', color='#c0392b')
ax.set_xlabel('Anomaly score')
ax.set_ylabel('Número de latidos')
ax.set_title('Distribución de anomaly scores por clase real')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nScore medio en limpios:    {scores[y_val == 0].mean():.4f}')
print(f'Score medio en artefactos: {scores[y_val == 1].mean():.4f}')
print(f'Ratio:                      {scores[y_val == 1].mean() / scores[y_val == 0].mean():.2f}x')

### 6.2 Curva ROC

La curva ROC enfrenta **TPR (True Positive Rate)** vs **FPR (False Positive Rate)** al variar el umbral sobre el anomaly score:

- **TPR = Recall = Sensibilidad**: de todos los artefactos reales, ¿cuántos detectamos?
- **FPR**: de todos los latidos limpios, ¿a cuántos marcamos por error?

**AUC (Area Under Curve)**:
- 1.0 → separación perfecta
- 0.5 → aleatorio (diagonal punteada)
- Interpretación probabilística: AUC = P(score de un artefacto > score de un latido limpio elegidos al azar)

In [ ]:
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, f1_score, precision_score, recall_score
)

fpr, tpr, thresholds = roc_curve(y_val, scores)
auc = roc_auc_score(y_val, scores)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'RCF (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatorio (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.15, color='#2563eb')

# Marcar percentiles del score como umbrales de interés
percentiles = [90, 95, 99]
for pct in percentiles:
    t_val = np.percentile(scores, pct)
    idx = np.argmin(np.abs(thresholds - t_val))
    ax.plot(fpr[idx], tpr[idx], 'o', color='#c0392b', markersize=9)
    ax.annotate(f'p{pct}={t_val:.2f}\nTPR={tpr[idx]:.2f}, FPR={fpr[idx]:.2f}',
                (fpr[idx], tpr[idx]), textcoords='offset points',
                xytext=(12, -5), fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#c0392b', alpha=0.9))

ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=11)
ax.set_title(f'Curva ROC — AUC = {auc:.3f}', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Curva Precision-Recall

Complemento crítico cuando la clase positiva es minoritaria (aquí ~10% de artefactos). Responde a "de los latidos que marco como artefacto, ¿cuántos realmente lo son?".

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_val, scores)
ap = average_precision_score(y_val, scores)
baseline = y_val.mean()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color='#27ae60', lw=2.5, label=f'RCF (AP = {ap:.3f})')
ax.axhline(baseline, color='k', linestyle='--', lw=1,
           label=f'Baseline (todo artefacto) = {baseline:.3f}')
ax.set_xlabel('Recall (cobertura de artefactos)', fontsize=11)
ax.set_ylabel('Precision (pureza de alertas)', fontsize=11)
ax.set_title(f'Curva Precision-Recall — AP = {ap:.3f}', fontsize=13)
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.4 Selección de umbral

RCF no produce probabilidades. Hay que elegir un umbral sobre el anomaly score. Tres criterios comunes:

1. **Percentil basado en la distribución**: marca el top 5% como anomalías (útil si conoces la tasa esperada).
2. **Máximo F1**: balancea precision y recall usando la etiqueta.
3. **Índice de Youden**: maximiza `TPR - FPR`, útil en medicina.

Para datos no etiquetados en producción, la opción 1 es la estándar. Aquí, como tenemos ground truth, comparamos las tres.

In [ ]:
# Barrer umbrales únicos del score
candidate_thresholds = np.unique(np.round(scores, 4))
f1_scores = []
for t in candidate_thresholds:
    y_pred_t = (scores >= t).astype(int)
    f1_scores.append(f1_score(y_val, y_pred_t, zero_division=0))

best_f1_idx = int(np.argmax(f1_scores))
t_f1 = candidate_thresholds[best_f1_idx]

# Índice de Youden
youden = tpr - fpr
t_youden = thresholds[int(np.argmax(youden))]

# Percentil 90 (tasa esperada de artefactos)
t_p90 = np.percentile(scores, 90)

print('=== Comparación de umbrales ===')
for name, t in [('F1 máximo', t_f1), ('Youden',  t_youden), ('Percentil 90', t_p90)]:
    y_pred_t = (scores >= t).astype(int)
    p = precision_score(y_val, y_pred_t, zero_division=0)
    r = recall_score(y_val, y_pred_t, zero_division=0)
    f = f1_score(y_val, y_pred_t, zero_division=0)
    print(f'  {name:15s} umbral={t:.3f}  precision={p:.3f}  recall={r:.3f}  F1={f:.3f}')

In [ ]:
# Visualización del barrido de umbrales
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(candidate_thresholds, f1_scores, color='#27ae60', lw=2, label='F1')
ax.axvline(t_f1, color='#27ae60', linestyle='--', alpha=0.6,
           label=f'Óptimo F1 = {t_f1:.3f}')
ax.axvline(t_youden, color='#f39c12', linestyle='--', alpha=0.6,
           label=f'Youden = {t_youden:.3f}')
ax.axvline(t_p90, color='#2563eb', linestyle='--', alpha=0.6,
           label=f'Percentil 90 = {t_p90:.3f}')
ax.set_xlabel('Umbral de anomaly score')
ax.set_ylabel('F1')
ax.set_title('F1 en función del umbral')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 Matriz de confusión

Con el umbral óptimo por F1.

In [ ]:
selected_threshold = t_f1
y_pred = (scores >= selected_threshold).astype(int)
cm = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
labels = np.array([
    [f'TN\n{tn}\n(limpio correcto)', f'FP\n{fp}\n(falsa alarma)'],
    [f'FN\n{fn}\n(artefacto no detectado)', f'TP\n{tp}\n(artefacto detectado)']
])
sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False,
            xticklabels=['Pred: Limpio', 'Pred: Artefacto'],
            yticklabels=['Real: Limpio', 'Real: Artefacto'], ax=ax)
ax.set_title(f'Matriz de confusión (umbral = {selected_threshold:.3f})', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Classification report ===')
print(classification_report(y_val, y_pred,
                             target_names=['Limpio (0)', 'Artefacto (1)']))

### 6.6 Top anomalías: inspección cualitativa

Miremos los 10 latidos con el score más alto. Idealmente todos son artefactos, y sus valores de features deberían ser visiblemente atípicos.

In [ ]:
val_df_scored = val_df.copy()
val_df_scored['anomaly_score'] = scores
val_df_scored['predicted'] = (scores >= selected_threshold).astype(int)

top10 = val_df_scored.nlargest(10, 'anomaly_score')[
    ['timestamp', 'rr_interval_ms', 'pr_interval_ms',
     'p_wave_amplitude_mv', 't_wave_amplitude_mv',
     'morphology_score', 'heart_rate_bpm',
     'anomaly_score', 'is_artifact', 'predicted']
]

print('TOP 10 latidos más anómalos:')
top10

### 6.7 Análisis temporal

Traza el anomaly score a lo largo del tiempo. Los artefactos deberían verse como picos.

In [ ]:
val_df_scored = val_df_scored.sort_values('time_offset_seconds').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
limpios = val_df_scored[val_df_scored['is_artifact'] == 0]
artefactos = val_df_scored[val_df_scored['is_artifact'] == 1]

ax.scatter(limpios['time_offset_seconds'], limpios['anomaly_score'],
           c='#27ae60', s=12, alpha=0.6, label='Limpio')
ax.scatter(artefactos['time_offset_seconds'], artefactos['anomaly_score'],
           c='#c0392b', s=30, alpha=0.9, label='Artefacto (real)')
ax.axhline(selected_threshold, color='black', linestyle='--', lw=1,
           label=f'Umbral = {selected_threshold:.3f}')
ax.set_xlabel('Tiempo (segundos desde inicio)')
ax.set_ylabel('Anomaly score')
ax.set_title('Anomaly score a lo largo del tiempo')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Interpretación y siguientes pasos

### Lectura del AUC en detección de anomalías
| AUC | Interpretación |
|---|---|
| 0.50 | Random — RCF no está capturando estructura |
| 0.60–0.75 | Pobre — revisar features y preprocesamiento |
| 0.75–0.90 | Aceptable para screening |
| > 0.90 | Buena separación; apto para producción con revisión humana sobre alertas |
| > 0.98 | Sospechoso de fuga (features que incluyen la etiqueta implícitamente) |

### Consideraciones para pasar a producción
1. **Datos reales > sintéticos**: este dataset es limpio por construcción. En ECG real habrá contaminación (latidos reales con ruido de fondo, saturación del amplificador).
2. **Normalización**: RCF es sensible a la escala de cada feature. Considera estandarizar antes de entrenar (ajustando mean/std con `StandardScaler`).
3. **Ventana temporal**: enriquecer con features agregadas de latidos adyacentes (ej: SDNN, RMSSD en una ventana de 30s).
4. **Reentrenamiento periódico**: el patrón "normal" cambia por paciente y por dispositivo. Programar reentrenamiento mensual o por paciente.
5. **Alertas graduadas**: en lugar de umbral binario, usar 3 niveles (verde / amarillo / rojo) con thresholds en percentiles 90/95/99.
6. **Ground truth continuo**: recolectar revisiones de cardiólogos sobre alertas y usarlas para validar el modelo regularmente.

### Alternativas a RCF
- **Isolation Forest** (scikit-learn): similar filosofía, corre localmente sin necesidad de endpoint.
- **Autoencoders**: aprenden a reconstruir señales limpias; el error de reconstrucción mide anomalía.
- **Modelos supervisados** (XGBoost, Random Forest): si llegas a tener etiquetas suficientes, suelen superar a los no supervisados.

## 8. Cleanup — eliminar endpoint

**Muy importante**: dejar el endpoint corriendo genera costos por hora. Siempre eliminarlo al final.

In [ ]:
predictor.delete_endpoint()
print('Endpoint eliminado.')